# AQIS lab: owner verification in Colab

Checks a work-package branch on a machine Claude does not control: the lab environment builds, the tests pass, the audit gate is clean, and the repository stays clean.

**What this does not replace:** `/wp-audit` reads the code against the spec and breaks code on purpose to prove tests catch it. In round 1 every test passed and the auditor still found three weak tests. Use this notebook *as well as* the audit, before you merge.

**Before you start:** a CPU runtime is enough (Runtime → Change runtime type → CPU). The `GITHUB_TOKEN` secret must be set (key icon on the left) with notebook access switched on.

Run the cells from top to bottom.

In [ ]:
# 1) Settings: edit these three lines, then run the cell
REPO   = "nasiraman5266/voice-bot-vad-harness"
BRANCH = "PASTE-THE-PR-BRANCH-NAME"   # on the pull request page: "... wants to merge ... from <this name>"
WP     = "WP-0.0"

import base64, os, shutil, subprocess
from google.colab import userdata
_token = userdata.get("GITHUB_TOKEN")
_header = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{_token}".encode()).decode()
WORK = "/content/verify"
shutil.rmtree(WORK, ignore_errors=True)
# The token is passed for this one command only; it is never written into .git/config.
subprocess.run(["git", "-c", f"http.extraHeader={_header}", "clone", "--quiet", "--branch", BRANCH,
                f"https://github.com/{REPO}.git", WORK], check=True)
os.chdir(WORK)
print("Checked out:", subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())
print("origin/main available:", subprocess.run(["git", "rev-parse", "--verify", "--quiet", "origin/main"],
                                               capture_output=True).returncode == 0)

In [ ]:
# 2) Build the environments exactly as the project documents them (takes a few minutes)
def sh(cmd, timeout=3600):
    proc = subprocess.run(cmd, shell=True, cwd=WORK, capture_output=True, text=True, timeout=timeout)
    return proc.returncode, proc.stdout + proc.stderr

setup_log = []
for name, cmd in [("install uv", "pip install -q uv"),
                  ("lab environment (dev_bootstrap.sh)", "bash aqis_lab/deploy/dev_bootstrap.sh")]:
    rc, out = sh(cmd)
    setup_log.append((name, rc, out))
if os.path.exists("aqis_lab/deploy/harness_provision.sh"):
    rc, out = sh("bash aqis_lab/deploy/harness_provision.sh")
    setup_log.append(("harness environment (harness_provision.sh)", rc, out))
for name, rc, out in setup_log:
    print(("PASS  " if rc == 0 else "FAIL  ") + name)
    if rc:
        print(out[-3000:])

In [ ]:
# 3) Run the checks and write the report
import datetime, re
PY = ".venv/bin/python"
FAST = "not slow and not gpu and not colab and not oracle and not network"
checks = [
    ("Version, must not say +dirty", f"{PY} -m aqis_lab version", lambda rc, out: rc == 0 and "+dirty" not in out),
    ("Fast lab tests", f'{PY} -m pytest tests/aqis_lab -q -p no:cacheprovider -m "{FAST}"', lambda rc, out: rc == 0),
    ("Slow lab tests", f"{PY} -m pytest tests/aqis_lab -q -p no:cacheprovider -m slow", lambda rc, out: rc in (0, 5)),
    ("Audit gate for this WP", f"{PY} -m tools.audit.gate --base origin/main --wp {WP}", lambda rc, out: rc == 0),
]
if os.path.exists(".venv-harness/bin/python"):
    checks.append(("Harness tests, compare with docs/baselines/harness_tests_baseline.md",
                   ".venv-harness/bin/python -m pytest tests -q -p no:cacheprovider --ignore=tests/aqis_lab", None))
checks.append(("Repository still clean afterwards", "git status --porcelain",
               lambda rc, out: rc == 0 and out.strip() == ""))

rows, details = [], []
for name, cmd, ok in checks:
    rc, out = sh(cmd)
    verdict = "INFO" if ok is None else ("PASS" if ok(rc, out) else "FAIL")
    lines = out.strip().splitlines()
    summary = next((l for l in reversed(lines) if re.search(r"passed|failed|error|finding|clean|dirty", l, re.I)),
                   lines[-1] if lines else "(no output)")
    rows.append(f"| {name} | {verdict} | `{cmd}` | {summary[:120].replace('|', '/')} |")
    details.append(f"### {name}\n\n```text\n{out[-4000:]}\n```")

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
rc_gpu, gpu = sh("nvidia-smi --query-gpu=name --format=csv,noheader")
report = "\n".join([
    f"# Owner verification in Colab: {WP}", "",
    f"- Branch: `{BRANCH}` at `{sh('git log -1 --format=%H')[1].strip()}`",
    f"- Checked: {stamp}",
    f"- Lab Python: {sh(PY + ' --version')[1].strip()}",
    f"- GPU: {gpu.strip() if rc_gpu == 0 else 'none (CPU runtime)'}", "",
    "| Check | Result | Command | Summary |", "| --- | --- | --- | --- |", *rows, "",
    "## Setup", *[f"- {n}: {'PASS' if r == 0 else 'FAIL'}" for n, r, _ in setup_log], "",
    "## Output", *details])
REPORT = f"/content/OWNER_VERIFY_COLAB_{WP}_{stamp}.md"
with open(REPORT, "w") as fh:
    fh.write(report)
print("\n".join(rows))
print("\nReport saved:", REPORT)

In [ ]:
# 4) Download the report to your computer
from google.colab import files
files.download(REPORT)

## What to do with the report

1. On GitHub, switch to the pull request's branch, open `docs/progress/<WP>/`, and choose **Add file → Upload files**. Upload the report and commit to that branch.
2. **All PASS:** note it on the pull request and merge when the audit says PASS and CI is green.
3. **Any FAIL:** in the work package's Claude session, type `/surgical-fix docs/progress/<WP>/<report file name>`. Do not debug inside Colab.